# 🎮 AIOS Quant RL Training (Stable-Baselines3)

Обучение Reinforcement Learning торгового агента в симулированной биржевой среде.

**T4 GPU**.

Агент решает, какую долю капитала держать в активе, максимизируя доходность и минимизируя просадки. Используем PPO из Stable-Baselines3 на среде, построенной по ценам из ccxt.

In [ ]:
!pip install -q ccxt pandas numpy gymnasium stable-baselines3
import ccxt, pandas as pd, numpy as np
print('✅ Зависимости установлены')

In [ ]:
# === YACHEIKA 2: Zagruzka cen (ustoichivaya, neskolko birzh s fallback) ===
import os, time, requests, pandas as pd, numpy as np

def _retry(fn, tries=4, delay=2):
    for i in range(tries):
        try:
            return fn()
        except Exception:
            if i == tries-1:
                raise
            time.sleep(delay)

def fetch_binance(sym, interval='1h', limit=2000):
    url=f"https://api.binance.com/api/v3/klines?symbol={sym}&interval={interval}&limit={limit}"
    d=_retry(lambda: requests.get(url,timeout=25).json())
    return [[int(x[0]),float(x[1]),float(x[2]),float(x[3]),float(x[4]),float(x[5])] for x in d]

def fetch_bybit(sym, interval='60', limit=2000):
    url=f"https://api.bybit.com/v5/market/kline?category=spot&symbol={sym}&interval={interval}&limit={limit}"
    d=_retry(lambda: requests.get(url,timeout=25).json())
    return [[int(float(x[0])),float(x[1]),float(x[2]),float(x[3]),float(x[4]),float(x[5])] for x in d.get('result',{}).get('list',[])]

def fetch_okx(sym, bar='1H', limit=2000):
    url=f"https://www.okx.com/api/v5/market/candles?instId={sym}&bar={bar}&limit={limit}"
    d=_retry(lambda: requests.get(url,timeout=25).json())
    rows=[[int(float(x[0])),float(x[1]),float(x[2]),float(x[3]),float(x[4]),float(x[5])] for x in d.get('data',[])]
    rows.sort(key=lambda r:r[0])
    return rows

def fetch_kraken(sym, interval=60, limit=2000):
    url=f"https://api.kraken.com/0/public/OHLC?pair={sym}&interval={interval}"
    d=_retry(lambda: requests.get(url,timeout=25).json())
    rows=[]
    for r in (d.get('result') or {}).values():
        if isinstance(r,list):
            rows=[[int(x[0]),float(x[1]),float(x[2]),float(x[3]),float(x[4]),float(x[5])] for x in r]
            break
    return rows[:limit]

def load_df():
    attempts = [
        ('binance','BTCUSDT'), ('okx','BTC-USDT'), ('bybit','BTCUSDT'), ('kraken','XXBTZUSD'),
    ]
    fetchers = {'binance':fetch_binance,'bybit':fetch_bybit,'okx':fetch_okx,'kraken':fetch_kraken}
    last=None
    for name, sym in attempts:
        try:
            rows = fetchers[name](sym)
            if len(rows) >= 500:
                df = pd.DataFrame(rows, columns=['ts','open','high','low','close','volume'])
                df = df.sort_values('ts').drop_duplicates('ts').reset_index(drop=True)
                df['returns'] = df['close'].pct_change().fillna(0)
                df['momentum'] = df['close'].pct_change(12).fillna(0)
                print('[OK] dannye s ' + name + ': ' + str(df.shape))
                return df
        except Exception as e:
            last=e
            print('  [skip] ' + name + ': ' + str(e)[:80])
    raise RuntimeError('Ne udalos zagruzit dannye: ' + repr(last))

df = load_df()
print('Ceny:', df.shape)


In [ ]:
# === ЯЧЕЙКА 3: RL-среда (gymnasium) ===
import gymnasium as gym
from gymnasium import spaces

class TradingEnv(gym.Env):
    def __init__(self, df, window=10):
        super().__init__()
        self.df = df.reset_index(drop=True)
        self.window = window
        self.action_space = spaces.Discrete(3)  # 0=полный выход, 1=50%, 2=100% в активе
        self.observation_space = spaces.Box(-np.inf, np.inf, (window*2,), dtype=np.float32)
        self.i = window
    def _obs(self):
        w = self.df['returns'].values[self.i-self.window:self.i]
        m = self.df['momentum'].values[self.i-self.window:self.i]
        return np.concatenate([w, m]).astype(np.float32)
    def reset(self, *, seed=None, options=None):
        self.i = self.window
        self.position = 0
        return self._obs(), {}
    def step(self, action):
        pos = action / 2.0
        r = self.df['returns'].values[self.i]
        reward = pos * r * 100  # доходность позиции (x100 для масштаба)
        self.position = pos
        self.i += 1
        done = self.i >= len(self.df) - 1
        return self._obs() if not done else self._obs(), float(reward), done, False, {}

env = TradingEnv(df)
print('✅ Среда создана')

In [ ]:
# === YACHEIKA 4: Obuchenie PPO ===
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv

vec = DummyVecEnv([lambda: TradingEnv(df)])
model = PPO('MlpPolicy', vec, verbose=0, learning_rate=1e-3, n_steps=256)
model.learn(total_timesteps=5000)
print('PPO_OBUCHEN')


In [ ]:
# === YACHEIKA 5: Validaciya agenta + sohranenie lokaly (bez mount) ===
import os
val = DummyVecEnv([lambda: TradingEnv(df.iloc[len(df)//2:].reset_index(drop=True))])
obs = val.reset()
total = 0
done = False
while not done:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, done, _ = val.step(action)
    total += reward
print(f"Itohovaya dohodnost agenta na validacii: {total:.2f}%")
os.makedirs('/content/models', exist_ok=True)
model.save('/content/models/ppo_trader.zip')
print("MODEL_SAVED", os.path.exists('/content/models/ppo_trader.zip'))
print("RL_DONE")
